# ✅ Soluciones — Clase 20: Árboles de decisión y Random Forest

Soluciones comentadas de los ejercicios de la Clase 20.

> Usa este archivo solo después de intentar los ejercicios por tu cuenta.

## Solución Ejercicio 1 — Árbol básico

In [ ]:
# Solución E1: árbol con max_depth=3, comparar train vs test
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

df = pd.read_csv("../../datasets/estudiantes.csv")
df["estado"] = ((df["evaluacion_python"] >= 60) & (df["asistencia_pct"] >= 70)).astype(int)

X = df[["asistencia_pct", "evaluacion_python", "evaluacion_pandas", "edad"]]
y = df["estado"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

arbol = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)

print(f"Train accuracy: {arbol.score(X_train, y_train):.3f}")
print(f"Test  accuracy: {arbol.score(X_test,  y_test):.3f}")
print("→ Si train >> test, hay sobreajuste. Con max_depth=3 suelen ser similares.")

## Solución Ejercicio 2 — Visualizar el árbol

In [ ]:
# Solución E2: graficar y leer el primer nodo
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

fig, ax = plt.subplots(figsize=(13, 6))
plot_tree(arbol, feature_names=X.columns.tolist(),
          class_names=["En Riesgo", "Aprobado"],
          filled=True, rounded=True, ax=ax, fontsize=9)
plt.tight_layout()
plt.show()
# El primer nodo (raíz) muestra la variable que mejor separa los datos.
# Generalmente será evaluacion_python o asistencia_pct.

## Solución Ejercicio 3 — Sobreajuste con max_depth

In [ ]:
# Solución E3: comparar tres árboles con distintas profundidades
for d in [1, 5, None]:
    a = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    print(f"max_depth={str(d):4s} → Train: {a.score(X_train, y_train):.3f}  Test: {a.score(X_test, y_test):.3f}")

print("\n→ Con max_depth=None: el árbol logra ~1.000 en train pero baja en test (memoriza).")

## Solución Ejercicio 4 — Random Forest e importancia

In [ ]:
# Solución E4: bosque, comparación de accuracy y gráfico de importancia
from sklearn.ensemble import RandomForestClassifier

bosque = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42).fit(X_train, y_train)

print(f"Árbol (d=3) Test:   {arbol.score(X_test, y_test):.3f}")
print(f"Bosque (100) Test:  {bosque.score(X_test, y_test):.3f}")

importancias = pd.Series(bosque.feature_importances_, index=X.columns).sort_values()
importancias.plot(kind="barh", color="forestgreen", figsize=(6, 3))
plt.title("Importancia de variables")
plt.tight_layout()
plt.show()
print("Variable más importante:", importancias.idxmax())

## Solución Ejercicio 5 — Predicción con probabilidades

In [ ]:
# Solución E5: predecir estado de estudiantes hipotéticos con predict_proba
nuevos = pd.DataFrame({
    "asistencia_pct":    [95, 40, 70],
    "evaluacion_python": [88, 35, 62],
    "evaluacion_pandas": [85, 30, 58],
    "edad":              [21, 18, 24]
})

clases = bosque.predict(nuevos)
probas = bosque.predict_proba(nuevos)

for i, (c, p) in enumerate(zip(clases, probas)):
    label = "Aprobado" if c == 1 else "En Riesgo"
    print(f"Estudiante {i+1}: {label}  | P(Aprobado)={p[1]*100:.1f}%  P(En Riesgo)={p[0]*100:.1f}%")